In [1]:
import pandas as pd

df_raw = pd.read_csv("club_signups.csv")
df = df_raw.copy()

print(df.shape)
print(df.dtypes)

(39, 9)
signup_id       object
student_id      object
full_name       object
email           object
faculty         object
city            object
club            object
signed_up_at    object
fee_paid        object
dtype: object


In [2]:
for col in ["faculty", "club", "city", "fee_paid"]:
    print(df[col].value_counts(dropna=False), "\n")

faculty
Pharmacy                                  7
EMS                                       7
IET                                       5
Management                                4
Information Engineering and Technology    4
iet                                       3
MET                                       3
Engineering and Materials Science         1
Pharma                                    1
Media Engineering and Technology          1
met                                       1
pharmacy                                  1
ems                                       1
Name: count, dtype: int64 

club
Robotics       8
Music          5
Chess          4
football       4
Debating       3
Football       3
Chess Club     3
Soccer         2
music          2
chess          2
debate club    1
Music          1
Debate         1
Name: count, dtype: int64 

city
Cairo         13
Alexandria    11
Giza           4
Alex           4
cairo          2
giza           2
alexandria     1
CAIRO         

In [3]:
def normalise(s):
    # strip ends, lower-case, collapse double spaces
    return s.str.strip().str.lower().str.split().str.join(" ")

for col in ["faculty", "club", "city"]:
    df[col] = normalise(df[col])

In [4]:
faculty_map = {
    "met": "MET", "media engineering and technology": "MET",
    "iet": "IET", "information engineering and technology": "IET",
    "ems": "EMS", "engineering and materials science": "EMS",
    "pharmacy": "Pharmacy", "pharma": "Pharmacy",
    "management": "Management",
}
club_map = {
    "robotics": "Robotics",
    "debate": "Debate", "debating": "Debate", "debate club": "Debate",
    "music": "Music",
    "football": "Football", "soccer": "Football",
    "chess": "Chess", "chess club": "Chess",
}
city_map = {
    "cairo": "Cairo",
    "giza": "Giza", "el giza": "Giza",
    "alexandria": "Alexandria", "alex": "Alexandria",
}

df["faculty"] = df["faculty"].map(faculty_map)
df["club"]    = df["club"].map(club_map)
df["city"]    = df["city"].map(city_map)

assert df["faculty"].isin({"MET", "IET", "EMS", "Pharmacy", "Management"}).all()
assert df["club"].isin({"Robotics", "Debate", "Music", "Football", "Chess"}).all()
assert df["city"].isin({"Cairo", "Giza", "Alexandria"}).all()

for col in ["faculty", "club", "city"]:
    print(df[col].value_counts(), "\n")

faculty
IET           12
Pharmacy       9
EMS            9
MET            5
Management     4
Name: count, dtype: int64 

club
Football    9
Chess       9
Music       8
Robotics    8
Debate      5
Name: count, dtype: int64 

city
Alexandria    16
Cairo         16
Giza           7
Name: count, dtype: int64 



In [5]:
df["full_name"] = df["full_name"].str.split().str.join(" ").str.title()
df["email"]     = df["email"].str.strip().str.lower()

# check: each student now has exactly one name and one email
print(df.groupby("student_id")[["full_name", "email"]].nunique().max())

full_name    1
email        1
dtype: int64


In [6]:
fee_map = {"yes": True, "y": True, "true": True, "1": True,
           "no": False, "n": False, "false": False, "0": False}

df["fee_paid"] = df["fee_paid"].astype(str).str.strip().str.lower().map(fee_map)

assert df["fee_paid"].notna().all()
df["fee_paid"] = df["fee_paid"].astype(bool)
print(df["fee_paid"].dtype)
print(df["fee_paid"].value_counts())

bool
fee_paid
True     27
False    12
Name: count, dtype: int64


In [7]:
s = df["signed_up_at"].str.strip()
iso   = s.str.match(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}$")
slash = s.str.match(r"^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$")

print("ISO rows:", iso.sum(), " slash rows:", slash.sum(), " all covered:", (iso | slash).all())
print("first number in slash format:", sorted(s[slash].str[:2].unique()))
print("second number in slash format:", s[slash].str[3:5].unique())

ISO rows: 25  slash rows: 14  all covered: True
first number in slash format: ['13', '14', '15', '16', '17', '18']
second number in slash format: ['09']


In [8]:
d_iso   = pd.to_datetime(s, format="%Y-%m-%d %H:%M", errors="coerce")
d_slash = pd.to_datetime(s, format="%d/%m/%Y %H:%M", errors="coerce")
df["signed_up_at"] = d_iso.fillna(d_slash)

assert df["signed_up_at"].notna().all()
print(df["signed_up_at"].dtype)
print(df["signed_up_at"].min(), "->", df["signed_up_at"].max())

datetime64[ns]
2026-09-13 14:36:00 -> 2026-09-18 23:36:00


In [9]:
n_start = len(df)
print("Exact duplicate rows:", df.duplicated().sum())
print(df[df.duplicated(keep=False)].sort_values("signup_id")[["signup_id", "full_name", "club"]])

df = df.drop_duplicates()
n_after_exact = len(df)
print(n_start, "->", n_after_exact)

Exact duplicate rows: 3
   signup_id   full_name      club
0       S015   Hana Zaki  Football
34      S015   Hana Zaki  Football
15      S022   Sara Zaki     Music
24      S022   Sara Zaki     Music
20      S023  Ziad Saleh     Chess
33      S023  Ziad Saleh     Chess
39 -> 36


In [10]:
dups = df[df.duplicated(["student_id", "club"], keep=False)]
print(dups.sort_values(["student_id", "club", "signed_up_at"])
          [["signup_id", "student_id", "full_name", "club", "signed_up_at", "fee_paid"]])

df = (df.sort_values("signed_up_at")
        .drop_duplicates(["student_id", "club"], keep="last"))
n_final = len(df)
print("Removed:", n_after_exact - n_final, "  Left:", n_final)

   signup_id student_id      full_name      club        signed_up_at  fee_paid
8       S020    55-9245    Ali Ibrahim     Music 2026-09-15 23:07:00     False
9       S033    55-9245    Ali Ibrahim     Music 2026-09-16 09:07:00      True
22      S021    61-4844   Mohamed Adel    Debate 2026-09-16 01:15:00     False
3       S035    61-4844   Mohamed Adel    Debate 2026-09-17 11:15:00      True
35      S017    61-7344  Laila Mostafa    Debate 2026-09-15 18:01:00     False
25      S034    61-7344  Laila Mostafa    Debate 2026-09-16 02:01:00      True
21      S032    61-7344  Laila Mostafa  Football 2026-09-17 23:36:00     False
38      S036    61-7344  Laila Mostafa  Football 2026-09-18 23:36:00      True
Removed: 4   Left: 32


In [11]:
assert not df.duplicated(["student_id", "club"]).any()

df = df.sort_values("signup_id").reset_index(drop=True)
df.to_csv("club_signups_clean.csv", index=False)
print(df.shape)
print(df.dtypes)
df

(32, 9)
signup_id               object
student_id              object
full_name               object
email                   object
faculty                 object
city                    object
club                    object
signed_up_at    datetime64[ns]
fee_paid                  bool
dtype: object


,signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
0,S001,58-9118,Mariam Kamel,mariam.kamel@student.guc.edu.eg,IET,Giza,Robotics,2026-09-13 14:36:00,True
1,S002,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg,IET,Alexandria,Debate,2026-09-13 15:38:00,True
2,S003,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Chess,2026-09-13 21:16:00,True
3,S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess,2026-09-13 22:36:00,True
4,S005,61-4733,Omar Adel,omar.adel@student.guc.edu.eg,EMS,Cairo,Football,2026-09-13 23:52:00,True
5,S006,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg,Pharmacy,Alexandria,Football,2026-09-14 01:22:00,True
6,S007,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg,Pharmacy,Alexandria,Robotics,2026-09-14 17:03:00,True
7,S008,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Music,2026-09-14 23:32:00,True
8,S009,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Robotics,2026-09-15 01:05:00,True
9,S010,61-3846,Ahmed Zaki,ahmed.zaki@student.guc.edu.eg,Management,Alexandria,Chess,2026-09-15 03:12:00,False


In [12]:
# What if we removed same-club duplicates BEFORE fixing spellings?
raw = df_raw.drop_duplicates()
print("Same-club duplicates found on raw data:", raw.duplicated(["student_id", "club"]).sum())

# Two different students with the same name
print(df[df["full_name"] == "Mohamed Adel"][["student_id", "faculty", "city", "club"]])

Same-club duplicates found on raw data: 2
   student_id faculty        city      club
1     55-2992     IET  Alexandria    Debate
7     61-4844     EMS       Cairo     Music
8     61-4844     EMS       Cairo  Robotics
20    55-2992     IET  Alexandria  Robotics
30    61-4844     EMS       Cairo    Debate
